In [ ]:
# Parameters
silver_schema = "silver"
gold_schema = "gold"
reset_trigger_config = False    # True: overwrite tuned thresholds in gold.trigger_config with the defaults below
enforce_scenario = True         # True: fail if the planted demo story (US stock-out, GB aging, GB->US pair...) is not detected
fail_on_check_error = True

# 04 - Gold: business-ready dimensions, facts, rebalancing mart, triggers and KPIs

UC1 *Athletic Marketplace Inventory Rebalancing* (customized deck slides 1-2; requirements sections 6.2, 6.3, 7).
One cell per Gold table; the layer is validated and only then written.

| Table | Purpose |
|---|---|
| `dim_product`, `dim_location`, `dim_market`, `dim_partner`, `dim_date` | conformed dimensions (`dim_market` carries accountable owners and owner gaps) |
| `fact_sell_through_daily` | units, full-price vs markdown units, net sales, COGS, gross margin (USD) |
| `fact_inventory_snapshot` | on hand, allocated, committed, in transit, available, weeks of cover |
| `fact_demand_signal_daily` | intent index, 7-day persistence, **SUSTAINED vs SPIKE** classification with event/weather reasons |
| `fact_inbound_supply`, `fact_wholesale_commitment`, `fact_returns_daily` | PO slips (T5), partner changes (T4), returns |
| `mart_rebalancing_candidates` | SKU-size x market: cover, days to stock-out, full-price sell-through, T1/T2 flags, markdown exposure |
| `mart_rebalancing_pairs` | T3: stock-out market x aging market with a DC-to-DC lane, lead time and margin guardrail |
| `mart_store_rebalancing` | Store view of each T1 SKU-size x market: per-store cover and role (SHORT / DONOR / BALANCED), store deficit and spare units for store-to-store moves |
| `trigger_config`, `trigger_event`, `kpi_snapshot` | tunable thresholds, fired triggers T1-T5, KPIs AN-01..AN-08 |
| `decision_*`, `watch_rule` | empty decision-lifecycle tables written later by the coworker agent |

Thresholds are read from `gold.trigger_config` when it exists, so the Regional Marketplace Director can tune them without code changes.

In [ ]:
import json, time
import datetime as _dt
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import numpy as np
import pandas as pd
import com.microsoft.spark.fabric   # registers spark.read.synapsesql (catalog-managed Databricks mirror)
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

LABEL = "SYNTHETIC"
# Default thresholds (requirements 6.3 / 7.2); seeded into gold.trigger_config on the first run
TRIGGER_CONFIG = [
    ("T1", "max_weeks_of_cover", 1.5, "weeks", "Projected stock-out: weeks of cover below this value"),
    ("T1", "min_intent_index", 1.3, "x baseline", "Projected stock-out: intent index at or above this value"),
    ("T2", "min_weeks_of_cover", 8.0, "weeks", "Aging inventory: weeks of cover above this value"),
    ("T2", "max_full_price_sell_through", 0.30, "ratio", "Aging inventory: full-price sell-through since launch below this value (calibrated from the deck's illustrative 25%)"),
    ("T3", "require_open_lane", 1.0, "flag", "Mismatch pair needs an active DC-to-DC transfer lane"),
    ("T4", "max_order_change_pct", 0.10, "ratio", "Wholesale change: cancelled or changed share of committed units"),
    ("T5", "max_po_slip_days", 5.0, "days", "Inbound slip: current promise minus original promise"),
    ("SIG", "sustained_min_days", 5.0, "days of last 7", "Sustained demand: days at or above the intent threshold"),
    ("SIG", "spike_jump_ratio", 2.0, "x prior 7-day mean", "Spike: single-day jump versus the prior 7-day mean index"),
    ("SIG", "soft_max_index", 0.7, "x baseline", "Soft demand: intent index at or below this value"),
    ("SIG", "conversion_confirm_ratio", 0.9, "ratio", "Sell-through confirms when last-7-day units >= ratio x prior 7 days"),
    ("SIG", "weather_temp_anomaly_c", 5.0, "deg C", "Weather flag: absolute temperature anomaly"),
    ("SIG", "weather_precip_ratio", 3.0, "x normal", "Weather flag: precipitation versus monthly normal (and > 10 mm)"),
    ("DQ", "stale_feed_hours", 36.0, "hours", "DQ-10 stale feed threshold"),
    ("INV", "rate_window_days", 14.0, "days", "Demand-rate window for weeks of cover"),
    ("INV", "target_weeks_of_cover", 4.0, "weeks", "Target cover used to size surplus and deficit units"),
    ("STORE", "short_max_weeks", 1.5, "weeks", "Store view: a store is SHORT below this cover on a SKU-size at risk (T1) in its market"),
    ("STORE", "donor_min_weeks", 4.5, "weeks", "Store view: a store can DONATE at or above this cover"),
    ("STORE", "donor_keep_weeks", 4.0, "weeks", "Store view: a donor always keeps this much of its own demand"),
    ("STORE", "display_min_units", 2.0, "units", "Store view: display stock a donor never gives away"),
    ("STORE", "bridge_days", 14.0, "days", "Store view: a short store is topped up to this many days while the DC refill is in transit"),
]
GOLD_KEYS = {
    "dim_product": ["sku_id"], "dim_location": ["location_id"], "dim_market": ["market_id"], "dim_partner": ["partner_id"],
    "dim_date": ["date_key"], "fact_sell_through_daily": ["business_date", "location_id", "sku_id"],
    "fact_inventory_snapshot": ["snapshot_date", "location_id", "sku_id"],
    "fact_demand_signal_daily": ["business_date", "market_id", "style_id", "color"],
    "fact_inbound_supply": ["po_id", "line_no"], "fact_wholesale_commitment": ["order_id", "line_no"],
    "fact_returns_daily": ["return_local_date", "location_id", "sku_id", "reason_code"],
    "mart_rebalancing_candidates": ["sku_id", "market_id"],
    "mart_rebalancing_pairs": ["sku_id", "source_market_id", "destination_market_id"],
    "mart_store_rebalancing": ["location_id", "sku_id"],
    "trigger_config": ["trigger_id", "parameter"], "trigger_event": ["trigger_id", "entity_key"], "kpi_snapshot": ["kpi_id", "market_id"],
}
# Decision-lifecycle tables: ERP decision history (HIST-*) is loaded by this notebook; the coworker agent appends its own
# rows, which every rebuild keeps. Fixed schema so the model, the ontology and the agent bind to stable types.
DECISION_TABLE_DDL = {
    "decision_case": "case_id STRING, trigger_id STRING, entity_key STRING, market_id STRING, sku_id STRING, opened_at TIMESTAMP, status STRING, owner_person_id STRING, severity STRING, summary STRING, record_label STRING",
    "decision_recommendation": "recommendation_id STRING, case_id STRING, option_id STRING, option_label STRING, units BIGINT, source_location_id STRING, destination_location_id STRING, projected_gm_usd DOUBLE, transfer_cost_usd DOUBLE, confidence DOUBLE, evidence_json STRING, created_at TIMESTAMP, record_label STRING",
    "decision_record": "decision_id STRING, case_id STRING, recommendation_id STRING, decided_by STRING, decision STRING, reason_code STRING, comment STRING, decided_at TIMESTAMP, record_label STRING",
    "decision_action": "action_id STRING, decision_id STRING, action_type STRING, tier INT, target_system STRING, payload_json STRING, status STRING, executed_at TIMESTAMP, record_label STRING",
    "decision_outcome": "outcome_id STRING, action_id STRING, measured_at TIMESTAMP, horizon_days INT, metric STRING, projected DOUBLE, actual DOUBLE, record_label STRING",
    "decision_thread_event": "event_id STRING, case_id STRING, author STRING, author_role STRING, message STRING, constraint_captured STRING, citations_json STRING, posted_at TIMESTAMP, record_label STRING",
    "watch_rule": "rule_id STRING, version INT, description STRING, condition_json STRING, constraints_json STRING, proposed_by STRING, approved_by STRING, status STRING, expires_at TIMESTAMP, fired_count BIGINT, record_label STRING",
}

DECISION_KEYS = {"decision_case": ["case_id"], "decision_recommendation": ["recommendation_id"], "decision_record": ["decision_id"],
                 "decision_action": ["action_id"], "decision_outcome": ["outcome_id"]}
HIST_PREFIX = "HIST-"
# ERP write-back status -> human-in-the-loop vocabulary (HL-02 APPROVE, HL-03 REVISE, HL-06 OVERRIDE, HL-07 REJECT)
ERP_TO_HL = {"APPROVED": "APPROVE", "MODIFIED": "REVISE", "REJECTED": "REJECT"}
EXECUTED_DECISIONS = ["APPROVE", "REVISE", "OVERRIDE"]


def write_decision_table(name, pdf):
    """Overwrite one decision table with the DDL types (history + preserved agent rows)."""
    spec = [c.split() for c in DECISION_TABLE_DDL[name].split(", ")]
    if pdf.empty:
        sdf = spark.createDataFrame([], DECISION_TABLE_DDL[name])
    else:
        sdf = to_spark(pdf[[c for c, _t in spec]]).select(*[F.col(c).cast(t).alias(c) for c, t in spec])
    sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{gold_schema}.{name}")
    return len(pdf)


def _label(df):
    df = df.drop(columns=[c for c in df.columns if c.startswith("_")], errors="ignore").copy()
    df["record_label"] = LABEL
    return df.reset_index(drop=True)


def _rates(st, as_of, days, by):
    win = st[(st.business_date >= as_of - pd.Timedelta(days=days)) & (st.business_date < as_of)]
    return (win.groupby(by).units_sold.sum() / days).rename(f"daily_rate_{days}d")

def to_pandas(sdf):
    """DECIMAL -> float, DATE -> datetime64 (midnight), TIMESTAMP -> tz-aware UTC."""
    cols, ts_cols = [], []
    for f in sdf.schema.fields:
        c = F.col(f"`{f.name}`")
        if isinstance(f.dataType, DecimalType):
            c = c.cast("double")
        elif isinstance(f.dataType, DateType):
            c = c.cast("timestamp")
        elif isinstance(f.dataType, TimestampType):
            ts_cols.append(f.name)
        cols.append(c.alias(f.name))
    pdf = sdf.select(*cols).toPandas()
    for c in ts_cols:
        pdf[c] = pd.to_datetime(pdf[c]).dt.tz_localize("UTC")
    return pdf


def _is_date_col(name, s):
    if not pd.api.types.is_datetime64_any_dtype(s) or isinstance(s.dtype, pd.DatetimeTZDtype):
        return False
    if not (name.endswith("_date") or name in ("date", "business_date", "fired_for_date", "as_of_date")):
        return False
    v = s.dropna()
    return v.empty or bool((v == v.dt.normalize()).all())


def to_spark(pdf):
    """pandas -> Spark with stable types: tz-aware -> UTC timestamp, midnight *_date columns -> DATE."""
    pdf = pdf.copy()
    date_cols = []
    for c in pdf.columns:
        s = pdf[c]
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            pdf[c] = s.dt.tz_convert("UTC").dt.tz_localize(None)
        elif _is_date_col(c, s):
            date_cols.append(c)
        elif s.dtype == object:
            kinds = {type(x) for x in s.dropna()}
            if not kinds:
                pdf[c] = s.astype("string")
            elif kinds <= {_dt.date}:
                pass
            elif kinds <= {Decimal}:
                pdf[c] = s.astype(float)
            elif kinds <= {pd.Timestamp, _dt.datetime}:
                pdf[c] = pd.to_datetime(s, utc=True).dt.tz_localize(None)
            elif kinds <= {bool}:
                pdf[c] = s.astype("boolean")
            elif not kinds <= {str}:
                pdf[c] = s.map(lambda x: None if x is None or (isinstance(x, float) and pd.isna(x)) else str(x)).astype("string")
    sdf = spark.createDataFrame(pdf)
    for c in date_cols:
        sdf = sdf.withColumn(c, F.to_date(F.col(c)))
    # pandas NaN/inf would land as IEEE NaN/Infinity, which the SQL endpoint and Direct Lake reject: store NULL
    for c, t in sdf.dtypes:
        if t in ("double", "float"):
            sdf = sdf.withColumn(c, F.when(F.isnan(c) | F.col(c).isin(float("inf"), float("-inf")), F.lit(None))
                                     .otherwise(F.col(c)))
    return sdf


def write_schema(schema, tables):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    rows = []
    for name, pdf in tables.items():
        to_spark(pdf).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{schema}.{name}")
        rows.append(dict(table_name=f"{schema}.{name}", rows=len(pdf)))
    return pd.DataFrame(rows)

# Check helpers: each check is a row (layer, table, check, passed, detail); assert_all fails the notebook
MAY_BE_EMPTY = {"mart_rebalancing_pairs", "mart_store_rebalancing", "trigger_event", "dq_quarantine", "regional_event",
                "erp_allocation_decision", "erp_stock_transfer_order", "decision_case", "decision_recommendation",
                "decision_record", "decision_action", "decision_outcome"}


def _r(layer, table, check, passed, detail=""):
    return dict(layer=layer, table_name=table, check=check, passed=bool(passed), detail=str(detail))


def _generic(layer, tables, keys):
    out = []
    for name, df in tables.items():
        out.append(_r(layer, name, "row_count", len(df) > 0 or name in MAY_BE_EMPTY, f"{len(df)} rows"))
        if "record_label" in df.columns:
            bad = int((df.record_label != LABEL).sum())
            out.append(_r(layer, name, "synthetic_label", bad == 0, f"{bad} unlabelled rows"))
        k = keys.get(name)
        if k and len(df):
            nulls = int(df[k].isna().any(axis=1).sum())
            dups = int(df.duplicated(k).sum())
            out.append(_r(layer, name, "key_not_null", nulls == 0, f"{nulls} null keys {k}"))
            out.append(_r(layer, name, "key_unique", dups == 0, f"{dups} duplicate keys {k}"))
    return out


def _fk(layer, child_name, child, col, parent, pcol=None):
    missing = int((~child[col].dropna().isin(parent[pcol or col])).sum())
    return _r(layer, child_name, f"fk_{col}", missing == 0, f"{missing} orphan {col}")


def assert_all(results):
    failed = results[~results.passed]
    if len(failed):
        raise AssertionError("Checks failed:\n" + failed.to_string(index=False))

In [ ]:
# Read Silver and the thresholds (tuned values in gold.trigger_config win over the defaults)
t0 = time.time()
names = [r.tableName for r in spark.sql(f"SHOW TABLES IN {silver_schema}").collect() if r.tableName != "dq_check_result"]
s = {n: to_pandas(spark.table(f"{silver_schema}.{n}")) for n in names}
c = {f"{t}.{p}": v for t, p, v, _u, _d in TRIGGER_CONFIG}
try:
    tuned = to_pandas(spark.table(f"{gold_schema}.trigger_config"))
except Exception:
    tuned = None                                    # first run: thresholds come from the defaults
if tuned is not None and len(tuned) and not reset_trigger_config:
    c.update({f"{r.trigger_id}.{r.parameter}": float(r.value) for r in tuned.itertuples()})
    print(f"using tuned thresholds from {gold_schema}.trigger_config")
as_of = pd.Timestamp(s["run_context"].as_of_date.iloc[0]).normalize()
g = {}
print(f"read {len(s)} Silver tables in {time.time() - t0:.0f}s; as_of {as_of.date()}")

In [ ]:
# gold.dim_product, dim_location, dim_market (with accountable owners and owner gaps), dim_partner, dim_date
g["dim_product"] = _label(s["product"][["sku_id", "style_id", "style_name", "model", "color_code", "color_name", "size", "size_run",
                                        "gtin", "category", "division", "gender", "lifecycle", "is_launch", "launch_date",
                                        "msrp_usd", "unit_cost_usd"]])
g["dim_location"] = _label(s["location"].drop(columns=["record_label"], errors="ignore"))
own = s["decision_owner"]
acc = own[own.role == "ACCOUNTABLE"].groupby("market_id").display_name.agg(lambda x: ", ".join(sorted(set(x))))
gaps = s["decision_owner_coverage"].groupby("market_id").dq_owner_gap.sum()
g["dim_market"] = _label(s["market"].drop(columns=["record_label"]).assign(
    accountable_owners=lambda d: d.market_id.map(acc), owner_gaps=lambda d: d.market_id.map(gaps).fillna(0).astype(int)))
g["dim_partner"] = _label(s["partner"].drop(columns=["record_label"]).merge(s["market"][["market_id", "region"]], on="market_id", how="left"))
dd = pd.DataFrame({"date": pd.date_range(s["sell_through_daily"].business_date.min(), as_of + pd.Timedelta(days=120), freq="D")})
dd["date_key"] = dd.date.dt.strftime("%Y%m%d").astype(int)
dd["year"], dd["quarter"], dd["month"] = dd.date.dt.year, dd.date.dt.quarter, dd.date.dt.month
dd["month_name"] = dd.date.dt.strftime("%b")
dd["iso_week"] = dd.date.dt.isocalendar().week.astype(int)
dd["day_of_week"] = dd.date.dt.day_name()
dd["is_weekend"] = dd.date.dt.weekday >= 5
dd["is_history"] = dd.date < as_of
dd["days_from_as_of"] = (dd.date - as_of).dt.days
g["dim_date"] = _label(dd)
print({k: len(v) for k, v in g.items()})

In [ ]:
# gold.fact_sell_through_daily: COGS, gross margin, markdown units; DQ-05 flag carried from returns
st = s["sell_through_daily"].merge(s["product"][["sku_id", "unit_cost_usd"]], on="sku_id", how="left")
flagged = s["returns"].loc[s["returns"].dq_returns_exceed_sales, ["location_id", "sku_id"]].drop_duplicates()
st = st.merge(flagged.assign(dq_returns_exceed_sales=True), on=["location_id", "sku_id"], how="left")
st["dq_returns_exceed_sales"] = st.dq_returns_exceed_sales.astype("boolean").fillna(False).astype(bool)
st["cogs_usd"] = (st.units_sold - st.units_returned) * st.unit_cost_usd
st["gross_margin_usd"] = st.net_sales_usd - st.cogs_usd
st["markdown_units"] = st.units_sold - st.full_price_units
g["fact_sell_through_daily"] = _label(st[["business_date", "location_id", "market_id", "channel", "sku_id", "style_id", "color",
                                          "size", "units_sold", "units_returned", "full_price_units", "markdown_units",
                                          "net_sales_local", "currency", "net_sales_usd", "cogs_usd", "gross_margin_usd",
                                          "is_restated", "dq_returns_exceed_sales", "source_batch_id"]])
print(f"gold.fact_sell_through_daily: {len(st):,}")

In [ ]:
# gold.fact_inventory_snapshot: demand rate (DCs use the market rate) and weeks of cover
inv = s["inventory_position"].copy()
st = s["sell_through_daily"]
win = int(c["INV.rate_window_days"])
inv = inv.merge(_rates(st, as_of, win, ["location_id", "sku_id"]).reset_index(), on=["location_id", "sku_id"], how="left")
inv = inv.merge(_rates(st, as_of, win, ["market_id", "sku_id"]).rename("market_daily_rate").reset_index(), on=["market_id", "sku_id"], how="left")
rate = np.where(inv.location_type.eq("DC"), inv.market_daily_rate, inv[f"daily_rate_{win}d"]).astype(float)
inv["demand_rate_daily"] = np.nan_to_num(rate, nan=0.0)
inv["weeks_of_cover"] = np.where(inv.demand_rate_daily > 0, inv.available_units / (inv.demand_rate_daily * 7), np.nan)
inv["no_recent_sales"] = inv.demand_rate_daily.eq(0)
inv["is_in_stock"] = inv.available_units > 0
g["fact_inventory_snapshot"] = _label(inv[["snapshot_date", "snapshot_ts", "location_id", "location_type", "market_id", "sku_id",
                                           "on_hand_units", "on_hand_units_raw", "allocated_units", "committed_wholesale_units",
                                           "in_transit_units", "available_units", "demand_rate_daily", "weeks_of_cover",
                                           "no_recent_sales", "is_in_stock", "dq_negative_stock", "dq_overcommit"]])
print(f"gold.fact_inventory_snapshot: {len(inv):,}")

In [ ]:
# gold.fact_demand_signal_daily: intent index vs peer markets + engagement, weather and events -> SUSTAINED vs SPIKE (7.2)
it = s["digital_intent_daily"][["business_date", "market_id", "style_id", "color", "search_volume", "pdp_views",
                                "add_to_cart", "notify_me"]].copy()
it["intent_score"] = it.search_volume + 0.3 * it.pdp_views + 3 * it.add_to_cart + 5 * it.notify_me
it = it[it.intent_score > 0]
ls = np.log(it.intent_score / it.groupby(["business_date", "market_id"]).intent_score.transform("sum"))
ls = ls - ls.groupby([it.business_date, it.style_id, it.color]).transform("median")      # vs same product elsewhere
ls = ls - ls.groupby([it.business_date, it.market_id]).transform("median")               # remove market mix effect
it["intent_index"] = np.exp(ls).round(4)

units = s["sell_through_daily"].groupby(["business_date", "market_id", "style_id", "color"]).units_sold.sum().rename("units_sold")
it = it.merge(units.reset_index(), on=["business_date", "market_id", "style_id", "color"], how="left")
it["units_sold"] = it.units_sold.fillna(0).astype("int64")
eng = s["member_engagement"][["business_date", "market_id", "style_id", "color", "app_product_opens", "wishlist_adds", "launch_reminders"]]
it = it.merge(eng, on=["business_date", "market_id", "style_id", "color"], how="left")

thr, k = c["T1.min_intent_index"], ["market_id", "style_id", "color"]
it = it.sort_values(k + ["business_date"]).reset_index(drop=True)
it["_above"] = (it.intent_index >= thr).astype(int)
grp = it.groupby(k, sort=True)
it["days_above_threshold_7d"] = grp.rolling("7D", on="business_date")._above.sum().to_numpy().astype(int)
it["intent_index_7d"] = grp.rolling("7D", on="business_date").intent_index.mean().to_numpy().round(4)
prior = grp.rolling("7D", on="business_date", closed="left").intent_index.mean().to_numpy()
it["jump_ratio"] = np.where(prior > 0, it.intent_index / prior, np.nan)
u7 = grp.rolling("7D", on="business_date").units_sold.sum().to_numpy()
u14 = grp.rolling("14D", on="business_date").units_sold.sum().to_numpy()
it["units_7d"], it["units_prev_7d"] = u7, u14 - u7
it["sell_through_confirms"] = it.units_7d >= c["SIG.conversion_confirm_ratio"] * it.units_prev_7d

it["category"] = it.style_id.map(s["product"].drop_duplicates("style_id").set_index("style_id").category)
ev_rows = []
for e in s["regional_event"].dropna(subset=["start_date", "end_date", "market_id"]).itertuples():
    for d in pd.date_range(e.start_date, e.end_date, freq="D"):
        ev_rows.append(dict(market_id=e.market_id, business_date=d, event_name=e.event_name, event_categories=str(e.categories or "ALL")))
ev = pd.DataFrame(ev_rows, columns=["market_id", "business_date", "event_name", "event_categories"])
it = it.merge(ev, on=["market_id", "business_date"], how="left")
hit = it.event_name.notna() & (it.event_categories.eq("ALL") |
                               pd.Series([str(ca) in str(ec).split(",") for ca, ec in zip(it.category, it.event_categories)], index=it.index))
it["event_flag"] = hit
it["event_name"] = it.event_name.where(hit)
it = it.drop(columns=["event_categories"]).sort_values(k + ["business_date", "event_name"]).drop_duplicates(k + ["business_date"])

wx = s["weather_daily"][["business_date", "market_id", "weather_status", "temp_anomaly_c", "precip_mm", "precip_ratio"]]
it = it.merge(wx, on=["business_date", "market_id"], how="left")
it["weather_status"] = it.weather_status.fillna("MISSING")
it["weather_flag"] = it.weather_status.eq("OBSERVED") & (
    (it.temp_anomaly_c.abs() >= c["SIG.weather_temp_anomaly_c"]) |
    ((it.precip_ratio >= c["SIG.weather_precip_ratio"]) & (it.precip_mm > 10)))
fresh = s["feed_freshness"]
it["dq_stale_feed"] = it.market_id.map(fresh[fresh.feed == "digital_intent_daily"].set_index("market_id").dq_stale_feed).fillna(True).astype(bool)

above = it.intent_index >= thr
sustained = (it.days_above_threshold_7d >= c["SIG.sustained_min_days"]) & it.sell_through_confirms
jump = (it.jump_ratio >= c["SIG.spike_jump_ratio"]) & (it.days_above_threshold_7d < c["SIG.sustained_min_days"])
spike = above & ~sustained & (it.event_flag | it.weather_flag | jump)
it["demand_class"] = np.select([sustained, spike, above, it.intent_index <= c["SIG.soft_max_index"]],
                               ["SUSTAINED", "SPIKE", "ELEVATED", "SOFT"], "NORMAL")
it["spike_reason"] = np.where(spike, np.where(it.event_flag, "EVENT", np.where(it.weather_flag, "WEATHER", "SINGLE_DAY_JUMP")), None)
it["demand_persistence_score"] = (it.days_above_threshold_7d / 7).round(3)
g["fact_demand_signal_daily"] = _label(it)
display(g["fact_demand_signal_daily"].demand_class.value_counts())

In [ ]:
# gold.fact_inbound_supply (T5: launch PO slipped more than the threshold)
po = s["purchase_order_line"].merge(s["product"][["sku_id", "is_launch"]], on="sku_id", how="left")
po["eta_date"] = po.current_promise_date
po["t5_inbound_slip"] = po.is_launch.fillna(False) & (po.slip_days > c["T5.max_po_slip_days"])
g["fact_inbound_supply"] = _label(po[["po_id", "line_no", "sku_id", "style_id", "vendor_id", "ship_to_location_id", "market_id",
                                      "ordered_units", "unit_cost_usd", "po_date", "original_promise_date", "current_promise_date",
                                      "eta_date", "slip_days", "po_status", "incoterm", "is_launch", "t5_inbound_slip"]])
print(f"gold.fact_inbound_supply: {len(po):,}, T5 lines {int(po.t5_inbound_slip.sum())}")

In [ ]:
# gold.fact_wholesale_commitment (T4: cancelled share of an order above the threshold)
ln = s["wholesale_order_line"].copy()
tot = ln[~ln.dq_cancel_resend].groupby("order_id").agg(order_units=("ordered_units", "sum"), order_cancelled=("cancelled_units", "sum"))
ln = ln.merge(tot.reset_index(), on="order_id", how="left")
ln["order_change_pct"] = np.where(ln.order_units > 0, ln.order_cancelled / ln.order_units, 0.0)
ln["t4_wholesale_change"] = ln.order_change_pct > c["T4.max_order_change_pct"]
g["fact_wholesale_commitment"] = _label(ln[["order_id", "line_no", "partner_id", "market_id", "ship_from_location_id", "sku_id",
                                            "ordered_units", "committed_units", "cancelled_units", "line_status", "effective_status",
                                            "order_status", "order_date", "requested_ship_date", "source_message_id",
                                            "order_change_pct", "t4_wholesale_change", "dq_cancel_resend"]])
print(f"gold.fact_wholesale_commitment: {len(ln):,}, T4 lines {int(ln.t4_wholesale_change.sum())}")

In [ ]:
# gold.fact_returns_daily
r = s["returns"]
fr = r.groupby(["return_local_date", "location_id", "market_id", "sku_id", "reason_code", "channel"], dropna=False).agg(
    return_lines=("return_id", "count"), units_returned=("units", "sum"), refund_local=("refund_local", "sum"),
    dq_returns_exceed_sales=("dq_returns_exceed_sales", "max")).reset_index()
fr["units_returned"] = fr.units_returned.astype("int64")
g["fact_returns_daily"] = _label(fr)
print(f"gold.fact_returns_daily: {len(fr):,}")

In [ ]:
# gold.mart_rebalancing_candidates: SKU-size x market cover, stock-out date, full-price sell-through, T1/T2 flags
inv = g["fact_inventory_snapshot"]
prod = s["product"].set_index("sku_id")
sell = inv.location_type.ne("DC")
agg = inv.assign(sell_avail=np.where(sell, inv.available_units, 0), dc_avail=np.where(sell, 0, inv.available_units),
                 sell_on_hand=np.where(sell, inv.on_hand_units, 0)).groupby(["sku_id", "market_id"]).agg(
    selling_available_units=("sell_avail", "sum"), dc_available_units=("dc_avail", "sum"),
    on_hand_units=("on_hand_units", "sum"), selling_on_hand_units=("sell_on_hand", "sum"),
    committed_wholesale_units=("committed_wholesale_units", "sum"), in_transit_units=("in_transit_units", "sum"),
    dq_overcommit=("dq_overcommit", "max"), dq_negative_stock=("dq_negative_stock", "max")).reset_index()
agg["total_available_units"] = agg.selling_available_units + agg.dc_available_units
st = s["sell_through_daily"]
r14 = _rates(st, as_of, int(c["INV.rate_window_days"]), ["market_id", "sku_id"]).rename("daily_rate").reset_index()
r7 = _rates(st, as_of, 7, ["market_id", "sku_id"]).rename("daily_rate_7d").reset_index()
agg = agg.merge(r14, on=["market_id", "sku_id"], how="left").merge(r7, on=["market_id", "sku_id"], how="left")
agg[["daily_rate", "daily_rate_7d"]] = agg[["daily_rate", "daily_rate_7d"]].fillna(0.0)
agg["weeks_of_cover"] = np.where(agg.daily_rate > 0, agg.total_available_units / (agg.daily_rate * 7), np.nan)
agg["selling_weeks_of_cover"] = np.where(agg.daily_rate > 0, agg.selling_available_units / (agg.daily_rate * 7), np.nan)
agg["days_to_stockout"] = np.where(agg.daily_rate > 0, agg.total_available_units / agg.daily_rate, np.nan)
for col in ("style_id", "color_code", "size", "category", "lifecycle", "is_launch", "launch_date", "unit_cost_usd", "msrp_usd"):
    agg[col] = agg.sku_id.map(prod[col])

since = st.merge(s["product"][["sku_id", "launch_date"]], on="sku_id")
since = since[since.business_date >= since.launch_date].groupby(["market_id", "sku_id"]).agg(
    units_since_launch=("units_sold", "sum"), full_price_units_since_launch=("full_price_units", "sum")).reset_index()
agg = agg.merge(since, on=["market_id", "sku_id"], how="left")
agg[["units_since_launch", "full_price_units_since_launch"]] = agg[["units_since_launch", "full_price_units_since_launch"]].fillna(0).astype("int64")
denom = agg.units_since_launch + agg.on_hand_units
agg["full_price_sell_through"] = np.where(denom > 0, agg.full_price_units_since_launch / denom, np.nan)

ds = g["fact_demand_signal_daily"]
latest = ds.sort_values("business_date").drop_duplicates(["market_id", "style_id", "color"], keep="last")
latest = latest.rename(columns={"color": "color_code", "business_date": "signal_date"})[
    ["market_id", "style_id", "color_code", "signal_date", "intent_index", "intent_index_7d", "demand_class",
     "spike_reason", "demand_persistence_score", "dq_stale_feed"]]
agg = agg.merge(latest, on=["market_id", "style_id", "color_code"], how="left")
agg["recent_spike_7d"] = agg.set_index(["market_id", "style_id", "color_code"]).index.isin(
    ds[(ds.business_date >= as_of - pd.Timedelta(days=7)) & ds.demand_class.eq("SPIKE")].set_index(["market_id", "style_id", "color"]).index)

price = s["price"].merge(s["market"][["market_id", "fx_usd_to_local"]], on="market_id")
price["full_price_usd"] = price.full_price_local / price.fx_usd_to_local
agg = agg.merge(price[["style_id", "market_id", "full_price_usd", "markdown_pct"]], on=["style_id", "market_id"], how="left")

target = c["INV.target_weeks_of_cover"] * 7
agg["deficit_units"] = np.ceil(np.clip(agg.daily_rate * target - agg.total_available_units, 0, None)).astype("int64")
agg["surplus_units"] = np.floor(np.clip(agg.total_available_units - agg.daily_rate * target, 0, None)).astype("int64")
agg["transferable_surplus_units"] = np.minimum(agg.surplus_units, agg.dc_available_units).astype("int64")
launch = agg.is_launch.fillna(False).astype(bool)
agg["t1_stockout_risk"] = launch & (agg.weeks_of_cover < c["T1.max_weeks_of_cover"]) & (agg.intent_index_7d >= c["T1.min_intent_index"])
agg["t2_aging_risk"] = launch & (agg.weeks_of_cover > c["T2.min_weeks_of_cover"]) & (agg.full_price_sell_through < c["T2.max_full_price_sell_through"])
agg["stockout_date_est"] = as_of + pd.to_timedelta(np.nan_to_num(agg.days_to_stockout, nan=9999).clip(max=9999), unit="D")
agg.loc[agg.days_to_stockout.isna(), "stockout_date_est"] = pd.NaT
agg["markdown_exposure_usd"] = np.where(agg.weeks_of_cover > c["T2.min_weeks_of_cover"],
                                        agg.on_hand_units * agg.full_price_usd * agg.markdown_pct.fillna(0.3), 0.0)
agg["rebalancing_role"] = np.select([agg.t1_stockout_risk, agg.t2_aging_risk], ["DESTINATION", "SOURCE"], "NONE")
agg["as_of_date"] = as_of
print(f"candidates: {len(agg):,}; T1 {int(agg.t1_stockout_risk.sum())}, T2 {int(agg.t2_aging_risk.sum())}")

In [ ]:
# gold.mart_rebalancing_pairs (T3): T1 in market A and T2 in market B for the same SKU-size, with an active DC -> DC lane B -> A
lanes = s["transfer_lane"]
lanes = lanes[lanes.from_location_type.eq("DC") & lanes.to_location_type.eq("DC") & lanes.is_cross_market]
guard = s["margin_guardrail"].set_index(["market_id", "category", "lifecycle"]).max_transfer_cost_pct
pairs = agg[agg.t1_stockout_risk].merge(agg[agg.t2_aging_risk], on="sku_id", suffixes=("_dst", "_src"))
rows = []
for p in pairs.itertuples():
    ln = lanes[(lanes.from_market_id == p.market_id_src) & (lanes.to_market_id == p.market_id_dst)]
    if ln.empty:
        continue
    best = ln.sort_values(["cost_per_unit_usd", "lead_days"]).iloc[0]
    units = int(min(p.deficit_units_dst, p.transferable_surplus_units_src))
    cap = guard.get((p.market_id_dst, p.category_dst, p.lifecycle_dst), np.nan)
    cost_pct = best.cost_per_unit_usd / p.full_price_usd_dst if p.full_price_usd_dst else np.nan
    rows.append(dict(
        sku_id=p.sku_id, source_market_id=p.market_id_src, destination_market_id=p.market_id_dst, style_id=p.style_id_dst,
        size=p.size_dst, from_location_id=best.from_location_id, to_location_id=best.to_location_id,
        lane_mode=best["mode"], lane_lead_days=int(best.lead_days), lane_cost_per_unit_usd=float(best.cost_per_unit_usd),
        proposed_units=units, destination_days_to_stockout=float(p.days_to_stockout_dst),
        source_weeks_of_cover=float(p.weeks_of_cover_src), destination_weeks_of_cover=float(p.weeks_of_cover_dst),
        arrives_before_stockout=bool(best.lead_days < p.days_to_stockout_dst),
        transfer_cost_pct_of_price=float(cost_pct), max_transfer_cost_pct=float(cap),
        within_margin_guardrail=bool(cost_pct <= cap) if pd.notna(cap) else None,
        transfer_cost_usd=round(units * float(best.cost_per_unit_usd), 2),
        full_price_revenue_protected_usd=round(units * float(p.full_price_usd_dst), 2), t3_mismatch_pair=True))
pairs_df = pd.DataFrame(rows, columns=[
    "sku_id", "source_market_id", "destination_market_id", "style_id", "size", "from_location_id", "to_location_id",
    "lane_mode", "lane_lead_days", "lane_cost_per_unit_usd", "proposed_units", "destination_days_to_stockout",
    "source_weeks_of_cover", "destination_weeks_of_cover", "arrives_before_stockout", "transfer_cost_pct_of_price",
    "max_transfer_cost_pct", "within_margin_guardrail", "transfer_cost_usd", "full_price_revenue_protected_usd", "t3_mismatch_pair"])
t3 = set(pairs_df.sku_id + "|" + pairs_df.destination_market_id) | set(pairs_df.sku_id + "|" + pairs_df.source_market_id)
agg["t3_mismatch_pair"] = (agg.sku_id + "|" + agg.market_id).isin(t3)
g["mart_rebalancing_candidates"], g["mart_rebalancing_pairs"] = _label(agg), _label(pairs_df)
display(g["mart_rebalancing_pairs"])

In [ ]:
# gold.mart_store_rebalancing: store view of each SKU-size x market at T1 risk (store extension of UC1).
# SHORT stores run out first; DONOR stores sell slowly and can spare units above their own 4 weeks plus display stock.
inv = g["fact_inventory_snapshot"]
mart = g["mart_rebalancing_candidates"]
t1k = mart[mart.t1_stockout_risk][["sku_id", "market_id", "style_id", "color_code", "size", "full_price_usd", "unit_cost_usd", "as_of_date"]]
loc = s["location"][["location_id", "location_name", "city", "format"]]
sr = inv[inv.location_type.eq("STORE")].merge(t1k, on=["sku_id", "market_id"]).merge(loc, on="location_id", how="left")
rate, av = sr.demand_rate_daily.astype(float), sr.available_units.astype(float)
sr["days_to_stockout"] = np.where(rate > 0, av / rate, np.nan)
short = (rate > 0) & (sr.weeks_of_cover < c["STORE.short_max_weeks"])
donor = (rate > 0) & (sr.weeks_of_cover >= c["STORE.donor_min_weeks"])
sr["store_role"] = np.select([short, donor], ["SHORT", "DONOR"], "BALANCED")
sr["deficit_units"] = np.where(short, np.ceil(np.clip(rate * c["STORE.bridge_days"] - av, 0, None)), 0).astype("int64")
spare = np.floor(av - rate * c["STORE.donor_keep_weeks"] * 7) - c["STORE.display_min_units"]
sr["surplus_units"] = np.where(donor, np.clip(spare, 0, None), 0).astype("int64")
g["mart_store_rebalancing"] = _label(sr[["location_id", "location_name", "city", "format", "market_id", "sku_id", "style_id",
                                         "color_code", "size", "demand_rate_daily", "available_units", "weeks_of_cover",
                                         "days_to_stockout", "store_role", "deficit_units", "surplus_units", "full_price_usd",
                                         "unit_cost_usd", "as_of_date"]])
print("mart_store_rebalancing:", len(sr), sr.store_role.value_counts().to_dict())


In [ ]:
# gold.trigger_config (effective thresholds, tunable) and gold.trigger_event (fired T1-T5)
g["trigger_config"] = _label(pd.DataFrame([dict(trigger_id=t, parameter=p, value=float(c.get(f"{t}.{p}", v)), unit=u, description=d)
                                           for t, p, v, u, d in TRIGGER_CONFIG]))
mart, prs = g["mart_rebalancing_candidates"], g["mart_rebalancing_pairs"]
ev = []


def add(tid, key, market, sku, severity, metrics):
    ev.append(dict(trigger_id=tid, entity_key=key, market_id=market, sku_id=sku, severity=severity,
                   metrics_json=json.dumps(metrics, default=lambda x: None if pd.isna(x) else float(x)), fired_for_date=as_of))


for r in mart[mart.t1_stockout_risk].itertuples():
    add("T1", f"{r.sku_id}|{r.market_id}", r.market_id, r.sku_id, "HIGH" if r.weeks_of_cover < 1 else "MEDIUM",
        dict(weeks_of_cover=r.weeks_of_cover, intent_index_7d=r.intent_index_7d, demand_class=r.demand_class))
for r in mart[mart.t2_aging_risk].itertuples():
    add("T2", f"{r.sku_id}|{r.market_id}", r.market_id, r.sku_id, "MEDIUM",
        dict(weeks_of_cover=r.weeks_of_cover, full_price_sell_through=r.full_price_sell_through))
for r in prs.itertuples():
    add("T3", f"{r.sku_id}|{r.source_market_id}->{r.destination_market_id}", r.destination_market_id, r.sku_id, "HIGH",
        dict(proposed_units=r.proposed_units, lane_lead_days=r.lane_lead_days, arrives_before_stockout=r.arrives_before_stockout))
wc = g["fact_wholesale_commitment"]
for oid, o in wc[wc.t4_wholesale_change].groupby("order_id"):
    add("T4", oid, o.market_id.iloc[0], None, "HIGH",
        dict(order_change_pct=o.order_change_pct.iloc[0], cancelled_units=o.cancelled_units.sum(), partner_id=o.partner_id.iloc[0]))
ib = g["fact_inbound_supply"]
for (po_id, mk), o in ib[ib.t5_inbound_slip].groupby(["po_id", "market_id"]):
    add("T5", po_id, mk, None, "MEDIUM", dict(slip_days=o.slip_days.max(), units=o.ordered_units.sum(), style_id=o.style_id.iloc[0]))
g["trigger_event"] = _label(pd.DataFrame(ev, columns=["trigger_id", "entity_key", "market_id", "sku_id", "severity", "metrics_json", "fired_for_date"]))
display(g["trigger_event"].groupby(["trigger_id", "market_id"]).size().rename("events").reset_index())

In [ ]:
# gold.decision_case / recommendation / record / action / outcome (DS-10 prior allocation decisions)
# ERP history (proposal_id HIST-*) is parsed from silver.allocation_decision.evidence_json and its stock transfer orders;
# rows the agent already wrote to the Gold decision tables are kept unchanged
def _ts(v):
    return pd.to_datetime(v, utc=True) if v else pd.NaT

ad = s.get("allocation_decision", pd.DataFrame(columns=["proposal_id"]))
sto = s.get("stock_transfer_order", pd.DataFrame(columns=["decision_id"]))
hist = {n: [] for n in DECISION_KEYS}
for r in ad[ad.proposal_id.astype(str).str.startswith(HIST_PREFIX)].itertuples():
    ev, case_id = json.loads(r.evidence_json), r.proposal_id
    cs, opts = ev["case"], {o["option_id"]: o for o in ev["options"]}
    opened = _ts(cs["opened_at"])
    rec_opt = ev.get("recommended_option_id", ev["chosen_option_id"])
    outcome = ev.get("outcome")
    hist["decision_case"].append(dict(
        case_id=case_id, trigger_id=cs["trigger_id"], entity_key=f"{cs['sku_id']}|{r.market_id}", market_id=r.market_id,
        sku_id=cs["sku_id"], opened_at=opened, owner_person_id=r.approver_person_id, severity=cs["severity"], summary=cs["summary"],
        status="CLOSED" if r.decision == "REJECTED" or (outcome and outcome.get("actual") is not None) else "MONITORING"))
    for o in ev["options"]:
        hist["decision_recommendation"].append(dict(
            recommendation_id=f"{case_id}-{o['option_id']}", case_id=case_id, option_id=o["option_id"],
            option_label=o["option_label"], units=int(o["units"]), source_location_id=o.get("source_location_id"),
            destination_location_id=o.get("destination_location_id"), projected_gm_usd=float(o["projected_gm_usd"]),
            transfer_cost_usd=float(o["transfer_cost_usd"]), confidence=float(o["confidence"]),
            evidence_json=json.dumps({"recommended": o["option_id"] == rec_opt, "source": "ERP_HISTORY"}),
            created_at=opened + pd.Timedelta(minutes=30)))
    decision, rec_id = ERP_TO_HL[r.decision], f"{case_id}-{rec_opt}"
    if decision == "REVISE":                                # HL-03: the revised scope is re-simulated as its own option
        o, k = opts[ev["chosen_option_id"]], ev["approved_units"] / opts[ev["chosen_option_id"]]["units"]
        rec_id = f"{rec_id}R"
        hist["decision_recommendation"].append(dict(
            recommendation_id=rec_id, case_id=case_id, option_id=f"{o['option_id']}R", option_label=f"{o['option_label']} (revised)",
            units=int(ev["approved_units"]), source_location_id=o.get("source_location_id"),
            destination_location_id=o.get("destination_location_id"), projected_gm_usd=round(o["projected_gm_usd"] * k, 2),
            transfer_cost_usd=round(o["transfer_cost_usd"] * k, 2), confidence=float(o["confidence"]),
            evidence_json=json.dumps({"revised_from": f"{case_id}-{o['option_id']}", "source": "ERP_HISTORY"}),
            created_at=r.decided_at_utc - pd.Timedelta(minutes=5)))
    hist["decision_record"].append(dict(
        decision_id=r.decision_id, case_id=case_id, recommendation_id=rec_id, decided_by=r.approver_person_id,
        decision=decision, reason_code=ev["reason_code"], comment=r.rationale, decided_at=r.decided_at_utc))
    if r.decision == "REJECTED":
        continue
    lines = sto[sto.decision_id == r.decision_id]
    if len(lines):
        units = int(lines.units.sum())
        act = dict(action_type="TRANSFER_RELEASE", target_system="WMS", executed_at=lines.created_at_utc.min(),
                   status="SUCCEEDED" if lines.sto_status.eq("RECEIVED").all() else "IN_PROGRESS",
                   payload_json=json.dumps({"sto_id": lines.sto_id.iloc[0], "units": units,
                                            "lines": lines[["sku_id", "units"]].to_dict("records")}))
    elif ev.get("action"):
        a = ev["action"]
        units = int(ev.get("approved_units", 0))
        act = dict(action_type=a["action_type"], target_system=a["target_system"], status=a["status"],
                   executed_at=_ts(a["executed_at"]), payload_json=json.dumps({"option_id": ev["chosen_option_id"]}))
    else:
        continue
    action_id = f"{case_id}-A1"
    hist["decision_action"].append(dict(action_id=action_id, decision_id=r.decision_id, tier=2, **act))
    if outcome:
        hist["decision_outcome"].append(dict(
            outcome_id=f"{case_id}-O1", action_id=action_id, measured_at=_ts(outcome["measured_at"]),
            horizon_days=int(outcome["horizon_days"]), metric=outcome["metric"], projected=float(outcome["projected"]),
            actual=np.nan if outcome["actual"] is None else float(outcome["actual"])))

# keep what the agent wrote (anything not from the ERP history)
hist_key = {"decision_case": "case_id", "decision_recommendation": "case_id", "decision_record": "case_id",
            "decision_action": "action_id", "decision_outcome": "outcome_id"}
dec = {}
for name, rows in hist.items():
    cols = [c.split()[0] for c in DECISION_TABLE_DDL[name].split(", ") if not c.startswith("record_label")]
    h = pd.DataFrame(rows, columns=cols)
    try:
        prev = to_pandas(spark.table(f"{gold_schema}.{name}"))
        prev = prev[~prev[hist_key[name]].astype(str).str.startswith(HIST_PREFIX)].drop(columns="record_label", errors="ignore")
    except Exception:
        prev = pd.DataFrame(columns=cols)
    dec[name] = _label(pd.concat([df for df in (h, prev) if len(df)], ignore_index=True) if len(h) or len(prev) else h)
print({n: len(d) for n, d in dec.items()})
display(dec["decision_record"][["case_id", "decision", "decided_by", "reason_code", "decided_at"]])


In [ ]:
# gold.kpi_snapshot (AN-01..AN-08; AN-04/07/08 from the decision tables, NO_DECISIONS for markets without any)
inv, mart = g["fact_inventory_snapshot"], g["mart_rebalancing_candidates"]
launch_skus = set(g["dim_product"].sku_id[g["dim_product"].is_launch])
dcase, drec = dec["decision_case"], dec["decision_record"]
dd = drec.merge(dcase[["case_id", "market_id", "opened_at"]], on="case_id", how="left")
dd["cycle_hours"] = (dd.decided_at - dd.opened_at).dt.total_seconds() / 3600
# AN-04 (same definition as the model measure): chosen recommendation GM$ - S0 GM$ of the case, over executed decisions
drc = dec["decision_recommendation"]
s0 = drc[drc.option_id == "S0"].set_index("case_id").projected_gm_usd
ex = dd[dd.decision.isin(EXECUTED_DECISIONS)].merge(drc[["recommendation_id", "projected_gm_usd"]], on="recommendation_id", how="left")
ex["gm_delta_usd"] = ex.projected_gm_usd - ex.case_id.map(s0).fillna(0.0)


def _dk(mk, value, has):
    return (value, "COMPUTED") if has else (np.nan, "NO_DECISIONS")


rows = []
for mk in sorted(g["dim_market"].market_id):
    dm = dd[dd.market_id == mk]
    an04 = _dk(mk, ex[ex.market_id == mk].gm_delta_usd.sum(), (ex.market_id == mk).any())
    an07 = _dk(mk, dm.cycle_hours.median(), len(dm) > 0)
    an08 = _dk(mk, dm.decision.eq("APPROVE").mean(), len(dm) > 0)
    i = inv[(inv.market_id == mk) & inv.location_type.ne("DC") & inv.sku_id.isin(launch_skus)]
    mm = mart[(mart.market_id == mk) & mart.is_launch.astype(bool)]
    fp = mm.full_price_units_since_launch.sum() / max(1, (mm.units_since_launch + mm.on_hand_units).sum())
    rows += [
        ("AN-01", "In-stock rate (launch SKU-sizes, selling locations)", mk, i.is_in_stock.mean() if len(i) else np.nan, "ratio", "COMPUTED"),
        ("AN-02", "Full-price sell-through since launch", mk, fp, "ratio", "COMPUTED"),
        ("AN-03", "Markdown exposure", mk, mart[mart.market_id == mk].markdown_exposure_usd.sum(), "USD", "COMPUTED"),
        ("AN-04", "Gross-margin protection (projected, chosen option vs S0)", mk, an04[0], "USD", an04[1]),
        ("AN-05", "Launch availability (any selling location)", mk, (mm.selling_available_units > 0).mean() if len(mm) else np.nan, "ratio", "COMPUTED"),
        ("AN-06", "Partner service level", mk, np.nan, "ratio", "PENDING_SHIPMENT_DATA"),
        ("AN-07", "Decision-cycle time (median, case opened to decided)", mk, an07[0], "hours", an07[1]),
        ("AN-08", "Recommendation quality (approved as recommended)", mk, an08[0], "ratio", an08[1]),
    ]
kpi = pd.DataFrame(rows, columns=["kpi_id", "kpi_name", "market_id", "value", "unit", "status"])
kpi["value"] = kpi.value.astype(float)
kpi["as_of_date"] = as_of
g["kpi_snapshot"] = _label(kpi)
display(kpi[kpi.status == "COMPUTED"].pivot(index="market_id", columns="kpi_id", values="value"))

In [ ]:
# Validation: counts, keys, duplicates, foreign keys and business rules (T1-T5, sustained vs spike)
out = _generic("gold", g, GOLD_KEYS)
out += _generic("gold", dec, DECISION_KEYS)
if len(dec["decision_case"]):
    dc_, dr_, da_, do_, rc_ = (dec[n] for n in ("decision_case", "decision_record", "decision_action", "decision_outcome", "decision_recommendation"))
    out += [
        _fk("gold", "decision_case", dc_, "sku_id", g["dim_product"]),
        _fk("gold", "decision_case", dc_, "market_id", g["dim_market"]),
        _fk("gold", "decision_recommendation", rc_, "case_id", dc_),
        _fk("gold", "decision_record", dr_, "case_id", dc_),
        _fk("gold", "decision_record", dr_, "recommendation_id", rc_),
        _fk("gold", "decision_action", da_, "decision_id", dr_),
        _fk("gold", "decision_outcome", do_, "action_id", da_),
        _r("gold", "decision_action", "AG-02_no_action_without_approval",
           da_.decision_id.isin(dr_[dr_.decision.isin(EXECUTED_DECISIONS)].decision_id).all(), "actions only for approved decisions"),
        _r("gold", "decision_record", "decided_after_opened",
           (dr_.merge(dc_[["case_id", "opened_at"]], on="case_id").pipe(lambda x: x.decided_at >= x.opened_at)).all(), ""),
    ]
for n in ("fact_sell_through_daily", "fact_inventory_snapshot", "mart_rebalancing_candidates"):
    out.append(_fk("gold", n, g[n], "sku_id", g["dim_product"]))
    out.append(_fk("gold", n, g[n], "market_id", g["dim_market"]))
out.append(_fk("gold", "fact_sell_through_daily", g["fact_sell_through_daily"], "location_id", g["dim_location"]))
out.append(_fk("gold", "fact_wholesale_commitment", g["fact_wholesale_commitment"], "partner_id", g["dim_partner"]))
mart, prs = g["mart_rebalancing_candidates"], g["mart_rebalancing_pairs"]
t1, t2 = mart[mart.t1_stockout_risk], mart[mart.t2_aging_risk]
out.append(_r("gold", "mart_rebalancing_candidates", "T1_rule",
              ((t1.weeks_of_cover < c["T1.max_weeks_of_cover"]) & (t1.intent_index_7d >= c["T1.min_intent_index"]) & t1.is_launch.astype(bool)).all(), f"{len(t1)} T1 rows"))
out.append(_r("gold", "mart_rebalancing_candidates", "T2_rule",
              ((t2.weeks_of_cover > c["T2.min_weeks_of_cover"]) & (t2.full_price_sell_through < c["T2.max_full_price_sell_through"])).all(), f"{len(t2)} T2 rows"))
out.append(_r("gold", "mart_rebalancing_candidates", "T1_T2_exclusive", not (mart.t1_stockout_risk & mart.t2_aging_risk).any(), "a SKU-market cannot be both"))
if len(prs):
    key = mart.set_index(["sku_id", "market_id"])
    dst_ok = key.loc[list(zip(prs.sku_id, prs.destination_market_id)), "t1_stockout_risk"].all()
    src_ok = key.loc[list(zip(prs.sku_id, prs.source_market_id)), "t2_aging_risk"].all()
    out.append(_r("gold", "mart_rebalancing_pairs", "T3_pair_sides", dst_ok and src_ok, f"{len(prs)} pairs"))
    out.append(_r("gold", "mart_rebalancing_pairs", "T3_units_positive_and_bounded",
                  ((prs.proposed_units > 0) & (prs.source_market_id != prs.destination_market_id)).all(), ""))
ws, ib, ds = g["fact_wholesale_commitment"], g["fact_inbound_supply"], g["fact_demand_signal_daily"]
out.append(_r("gold", "fact_wholesale_commitment", "T4_rule", (ws[ws.t4_wholesale_change].order_change_pct > c["T4.max_order_change_pct"]).all(), ""))
out.append(_r("gold", "fact_inbound_supply", "T5_rule", (ib[ib.t5_inbound_slip].slip_days > c["T5.max_po_slip_days"]).all(), ""))
sus = ds[ds.demand_class == "SUSTAINED"]
out.append(_r("gold", "fact_demand_signal_daily", "sustained_needs_persistence_and_conversion",
              ((sus.days_above_threshold_7d >= c["SIG.sustained_min_days"]) & sus.sell_through_confirms).all(), ""))
out.append(_r("gold", "fact_demand_signal_daily", "spike_has_reason", ds[ds.demand_class == "SPIKE"].spike_reason.notna().all(), ""))
fs = g["fact_sell_through_daily"]
out.append(_r("gold", "fact_sell_through_daily", "full_price_plus_markdown_eq_units", (fs.full_price_units + fs.markdown_units == fs.units_sold).all(), ""))
te = g["trigger_event"]
out.append(_r("gold", "trigger_event", "triggers_match_flags", (te.trigger_id == "T1").sum() == len(t1) and (te.trigger_id == "T2").sum() == len(t2), ""))
sm = g["mart_store_rebalancing"]
if len(sm):
    out.append(_fk("gold", "mart_store_rebalancing", sm, "location_id", g["dim_location"]))
    out.append(_r("gold", "mart_store_rebalancing", "only_T1_sku_markets",
                  sm.set_index(["sku_id", "market_id"]).index.isin(t1.set_index(["sku_id", "market_id"]).index).all(), ""))
    dn = sm[(sm.store_role == "DONOR") & (sm.surplus_units > 0)]
    out.append(_r("gold", "mart_store_rebalancing", "donor_keeps_4_weeks_plus_display",
                  ((dn.available_units - dn.surplus_units) >= dn.demand_rate_daily * c["STORE.donor_keep_weeks"] * 7
                   + c["STORE.display_min_units"] - 1e-9).all(), f"{len(dn)} donor rows"))
    out.append(_r("gold", "mart_store_rebalancing", "units_non_negative_and_role_consistent",
                  (sm.surplus_units >= 0).all() and (sm.deficit_units >= 0).all()
                  and (sm[sm.store_role != "DONOR"].surplus_units == 0).all() and (sm[sm.store_role != "SHORT"].deficit_units == 0).all(), ""))
checks = pd.DataFrame(out)

# Planted demo story (requirements 7.4)
hero = "HM9443"
h = mart[mart.style_id == hero]
scenario = pd.DataFrame([
    _r("scenario", "trigger_event", "US_hero_T1", h[(h.market_id == "US") & h.t1_stockout_risk].shape[0] > 0),
    _r("scenario", "trigger_event", "GB_hero_T2", h[(h.market_id == "GB") & h.t2_aging_risk].shape[0] > 0),
    _r("scenario", "mart_rebalancing_pairs", "GB_to_US_T3", ((prs.source_market_id == "GB") & (prs.destination_market_id == "US")).any()),
    _r("scenario", "fact_demand_signal_daily", "JP_event_spike",
       ds[(ds.style_id == hero) & (ds.market_id == "JP") & (ds.business_date == pd.Timestamp("2026-10-24"))].spike_reason.eq("EVENT").any(),
       "Tokyo City Run Expo 2026-10-24"),
    _r("scenario", "mart_store_rebalancing", "US_hero_store_donors_and_short",
       ((sm.style_id == hero) & (sm.market_id == "US") & (sm.store_role == "DONOR") & (sm.surplus_units > 0)).any()
       and ((sm.style_id == hero) & (sm.market_id == "US") & (sm.store_role == "SHORT")).any(),
       "slow-selling US stores can spare hero units for stores about to run out"),
    _r("scenario", "fact_demand_signal_daily", "US_hero_sustained", ds[(ds.style_id == hero) & (ds.market_id == "US")].demand_class.eq("SUSTAINED").any()),
    _r("scenario", "feed_freshness", "BR_intent_stale",
       s["feed_freshness"].query("feed == 'digital_intent_daily' and market_id == 'BR'").dq_stale_feed.any()),
    _r("scenario", "trigger_event", "T4_fired", (te.trigger_id == "T4").any()),
    _r("scenario", "trigger_event", "T5_fired", (te.trigger_id == "T5").any()),
    _r("scenario", "decision_owner_coverage", "owner_gap_flagged", s["decision_owner_coverage"].dq_owner_gap.any()),
])
print(f"{int(checks.passed.sum())}/{len(checks)} checks passed; scenario {int(scenario.passed.sum())}/{len(scenario)}")
display(pd.concat([checks[~checks.passed], scenario], ignore_index=True))
if fail_on_check_error:
    assert_all(checks)
if enforce_scenario:
    assert_all(scenario)

In [ ]:
# Write Gold only after every check passed; decision tables keep agent rows and refresh the ERP history
t0 = time.time()
written = write_schema(gold_schema, {**g, "dq_check_result": pd.concat([checks, scenario], ignore_index=True).assign(record_label=LABEL)})
dec_rows = {name: write_decision_table(name, pdf) for name, pdf in dec.items()}
for name, cols in DECISION_TABLE_DDL.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_schema}.{name} ({cols}) USING DELTA")
print(f"wrote {len(written)} Gold tables and decision tables {dec_rows} in {time.time() - t0:.0f}s")
display(written)